# Notebook 2 — Dot plot: chrF++ a nivel segmento vs juicio humano

Cruza el **chrF++ por segmento** del sistema evaluado (`v2-nostrat`) con los
**juicios humanos** de una lingüista con experiencia en qom.

### Los datos de juicio humano

- Segmentos evaluados en **ambas direcciones** (`qom2es` y `es2qom`), en un mismo CSV
  (una fila por segmento y dirección; la dirección va en la columna `direction`).
- Las oraciones **se eligieron sin garantizar que vinieran de un split de test**
  controlado, así que su `segment_id` se recupera por texto contra el corpus `qomL-hf`
  completo, y la hipótesis se toma de la columna con la predicción del sistema.
- Juicios **categóricos** (aproximadamente: correcto / incorrecto / dudoso).
- **No siguen el esquema MQM**: la tipificación MQM es una relectura posterior, así que
  **no hay severidades ni puntajes numéricos**.

> **No** se inventa un escalar de calidad ni se deriva uno a partir de las categorías.
> El origen no controlado de los segmentos es una limitación a declarar en la lectura.

### La pregunta de la figura

¿Los segmentos juzgados **incorrectos** se separan de los **correctos** en el eje de
chrF++, o se mezclan?

## Celda de configuración

Requiere que la **Notebook 1** haya corrido antes (produce `results/chrf_por_segmento.csv`
y `results/traducciones_tidy.csv`). **Completá `JUICIOS_CSV`** (columnas `segment_id,
direction, juicio_humano`) y, si querés, fijá `SISTEMA_EVALUADO`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

RANDOM_STATE = 20260729
np.random.seed(RANDOM_STATE)

# ── Entradas ──────────────────────────────────────────────────────────────────
DATA_DIR = Path("data")
# CSV de juicios. Puede NO traer segment_id: se recupera por texto desde qomL-hf.
# Columnas mínimas: direction, juicio (+ texto qom/es para recuperar el id).
JUICIOS_CSV = DATA_DIR / "human_eval/v2-nostrat.csv"

# Corpus completo empaquetado (parquet por config y split). De acá se recuperan
# los segment_id de TODOS los segmentos (no sólo del test común de Base).
QOML_HF_DIR = DATA_DIR / "qomL-hf"

# Salidas de la Notebook 1 (opcionales acá: sólo se usan como atajo si existen).
RESULTS_DIR = Path("data/results")
SEGMENT_CHRF_CSV = RESULTS_DIR / "chrf_por_segmento.csv"
TIDY_CSV         = RESULTS_DIR / "traducciones_tidy.csv"

FIG_DIR = Path("poster/figures")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# n mínimo por categoría para calcular correlaciones/tests.
N_MINIMO_TESTS = 10
N_BOOTSTRAP = 100

# Sistema cuyas salidas evaluó la lingüista.
SISTEMA_EVALUADO = "qom-mt-v2-aleatorio"

# ── Hipótesis del sistema evaluado ────────────────────────────────────────────
# Las oraciones evaluadas pueden NO estar en el test común, así que no siempre hay
# chrF++ precalculado. La hipótesis se obtiene, en este orden de preferencia:
#
#   1) COLUMNA INLINE del CSV de juicios: si el CSV ya trae la predicción del
#      sistema (p. ej. la columna "v2-nostrat"), se usa esa. Es lo más fiel:
#      es exactamente el texto que vio la lingüista. None = autodetectar.
COL_PREDICCION_INLINE = None
#
#   2) CORRER EL MODELO: si no hay columna inline, se generan las hipótesis con el
#      checkpoint de v2-nostrat sobre el texto fuente. Completá el/los checkpoint(s).
CHECKPOINT_EVALUADO = None    # str (un modelo p/ambas direcciones) o
                              # {"qom2es": "ruta/ckpt", "es2qom": "ruta/ckpt"}
GEN_PARAMS = dict(num_beams=4, no_repeat_ngram_size=3, max_new_tokens=128)
LANG = {"qom": "grn_Latn", "es": "spa_Latn"}
DIRECTION_LANGS = {"qom2es": ("qom", "es"), "es2qom": ("es", "qom")}

print("Config Notebook 2 cargada. Sistema evaluado:", SISTEMA_EVALUADO)

## 2.1 — Datos: hipótesis, referencia y chrF++ por segmento

El CSV trae, por fila: `direction`, el **texto fuente** y la **referencia** (columnas
`text_source` / `text_reference`), la **predicción** del sistema evaluado (columna
`v2-nostrat`) y el `juicio` de la lingüista. Con eso:

1. **chrF++ por segmento** se calcula directo: `chrF++(predicción, text_reference)`.
   No depende de que el segmento esté en el test común ni en `chrf_por_segmento.csv`.
2. **`segment_id`** se recupera contra `qomL-hf` **solo para trazabilidad** (linkear cada
   oración al corpus): es **no bloqueante**, si no matchea la fila igual entra al dot plot.

> Compatibilidad: si el CSV todavía usa `qom`/`es` en vez de `text_source`/`text_reference`,
> la notebook lo detecta y resuelve el orden probando ambas orientaciones (en `es2qom` esas
> columnas venían invertidas). Lo recomendado es usar `text_source`/`text_reference`.

> Limitación: los segmentos no salen de un split de test controlado. Se declara en la lectura.

In [ ]:
import re
import unicodedata

# ── Normalización de direcciones ──────────────────────────────────────────────
DIRECTION_ALIASES = {
    "qom2es": "qom2es", "qom-es": "qom2es", "qom_es": "qom2es", "qomes": "qom2es",
    "qom->es": "qom2es", "qom→es": "qom2es", "tob2spa": "qom2es",
    "es2qom": "es2qom", "es-qom": "es2qom", "es_qom": "es2qom", "esqom": "es2qom",
    "es->qom": "es2qom", "es→qom": "es2qom", "spa2tob": "es2qom",
}
def norm_direction(v):
    k = str(v).strip().lower().replace(" ", "")
    if k in DIRECTION_ALIASES:
        return DIRECTION_ALIASES[k]
    raise ValueError(f"Dirección no reconocida: {v!r}. Agregala a DIRECTION_ALIASES.")

_PUNT = re.compile(r"[^\w\s]", flags=re.UNICODE)
_ESP  = re.compile(r"\s+")
def normalizar_texto(t):
    t = unicodedata.normalize("NFC", str(t)).lower()
    t = _PUNT.sub(" ", t); t = _ESP.sub(" ", t).strip()
    return t

CATEGORIAS_ESPERADAS = {"correcto", "incorrecto", "dudoso"}
CAT_ALIASES = {
    "correcto": "correcto", "correcta": "correcto", "ok": "correcto", "bien": "correcto",
    "si": "correcto", "sí": "correcto", "1": "correcto",
    "incorrecto": "incorrecto", "incorrecta": "incorrecto", "mal": "incorrecto",
    "no": "incorrecto", "0": "incorrecto",
    "dudoso": "dudoso", "dudosa": "dudoso", "duda": "dudoso", "?": "dudoso",
    "parcial": "dudoso",
}
def norm_categoria(v):
    return CAT_ALIASES.get(str(v).strip().lower(), str(v).strip().lower())

def _candidatos_col_pred(sistema):
    s = sistema.lower().replace("qom-mt-", "").replace("qom_mt_", "")
    s = s.replace("aleatorio", "nostrat").replace("estratificado", "strat")
    out, vistos = [], set()
    for c in [sistema, s, s.replace("-", "_"), s.replace("_", "-")]:
        if c and c not in vistos: vistos.add(c); out.append(c)
    return out

# ── Carga del CSV de juicios ──────────────────────────────────────────────────
jr = pd.read_csv(JUICIOS_CSV)
lower = {c.lower().strip(): c for c in jr.columns}
def _col(*cands, req=True):
    for c in cands:
        if c in lower: return lower[c]
    if req: raise ValueError(f"Falta alguna de estas columnas en {JUICIOS_CSV}: {cands}")
    return None

col_dir    = _col("direction", "dir", "sentido", "direccion", "dirección")
col_juicio = _col("juicio_humano", "juicio", "categoria", "categoría", "label",
                  "evaluacion", "evaluación")
# Preferimos columnas por ROL (fuente/referencia); si no, caemos a qom/es.
col_ts = _col("text_source", "source_text", "texto_fuente", "fuente", "src", req=False)
col_tr = _col("text_reference", "reference_text", "texto_referencia", "referencia",
              "reference", "ref", req=False)
col_qom = _col("qom", "toba", "tob", req=False)
col_es  = _col("es", "spa", "español", "espanol", "castellano", req=False)

j = pd.DataFrame({
    "direction":     jr[col_dir].map(norm_direction),
    "juicio_humano": jr[col_juicio].map(norm_categoria),
})
if col_ts is not None and col_tr is not None:
    MODO_TEXTO = "rol"          # text_source = fuente, text_reference = referencia
    j["text_source"]    = jr[col_ts].astype("string")
    j["text_reference"] = jr[col_tr].astype("string")
elif col_qom is not None or col_es is not None:
    MODO_TEXTO = "idioma"       # columnas qom/es (posible swap en es2qom)
    if col_qom is not None: j["qom"] = jr[col_qom].astype("string")
    if col_es  is not None: j["es"]  = jr[col_es].astype("string")
else:
    MODO_TEXTO = "ninguno"      # sin texto: sólo sirve si el CSV ya trae segment_id
if _col("segment_id", "seg_id", req=False):
    j["segment_id"] = jr[_col("segment_id", "seg_id")]

# Predicción inline del sistema evaluado.
if COL_PREDICCION_INLINE is not None:
    key = COL_PREDICCION_INLINE.lower().strip()
    if key not in lower:
        raise ValueError(f"No encuentro la columna inline '{COL_PREDICCION_INLINE}'. "
                         f"Columnas: {list(jr.columns)}")
    col_pred = lower[key]
else:
    col_pred = None
    for c in _candidatos_col_pred(SISTEMA_EVALUADO):
        if c in lower: col_pred = lower[c]; break
if col_pred is not None:
    j["hyp_inline"] = jr[col_pred].astype("string")

inesperadas = set(j["juicio_humano"]) - CATEGORIAS_ESPERADAS
if inesperadas:
    print(f"[aviso] Categorías fuera de {CATEGORIAS_ESPERADAS}: {inesperadas}.")
print("Filas de juicio:", len(j), "| modo de texto:", MODO_TEXTO)
print(j.groupby(["direction", "juicio_humano"]).size().to_string())
print("Columna de predicción inline:", col_pred if col_pred else "(no hay → correr modelo)")

In [ ]:
# ── Recuperar segment_id desde qomL-hf (TRAZABILIDAD, no bloqueante) ──────────
from difflib import SequenceMatcher

UMBRAL_DIFUSO = 0.90
RECUPERAR_ID = True     # poné False para saltear el matching contra qomL-hf

def _config_preferida(sistema):
    s = sistema.lower().replace("qom-mt-", "").replace("qom_mt_", "")
    return s.replace("aleatorio", "nostrat").replace("estratificado", "strat")

def _indice_qomL_hf():
    parquets = sorted(QOML_HF_DIR.glob("*/*.parquet"))
    if not parquets:
        raise FileNotFoundError(f"No hay parquets en {QOML_HF_DIR}/*/*.parquet.")
    trozos = []
    for p in parquets:
        df = pd.read_parquet(p)
        if not {"qom", "es"} <= set(df.columns):
            print(f"[aviso] {p} sin columnas qom/es: se saltea."); continue
        sub = df[["qom", "es"] + (["id"] if "id" in df.columns else [])].copy()
        sub["hf_id"]  = sub["id"] if "id" in sub.columns else pd.NA
        sub["config"] = p.parent.name; sub["split"] = p.stem
        trozos.append(sub[["hf_id", "config", "split", "qom", "es"]])
    idx = pd.concat(trozos, ignore_index=True).dropna(subset=["qom", "es"])
    idx["qom_norm"] = idx["qom"].map(normalizar_texto)
    idx["es_norm"]  = idx["es"].map(normalizar_texto)
    pref = _config_preferida(SISTEMA_EVALUADO)
    idx["_pref"] = (~idx["config"].str.contains(pref, case=False, na=False)).astype(int)
    idx = (idx.sort_values(["_pref", "config", "split"])
              .drop_duplicates(["qom_norm", "es_norm"], keep="first").reset_index(drop=True))
    idx["segment_id"] = [r.hf_id if pd.notna(r.hf_id) else f"{r.config}:{r.split}:{i}"
                         for i, r in enumerate(idx.itertuples(index=False))]
    return idx

_PRIO = {"par_exacto": 3, "solo_qom": 2, "solo_es": 2, "difuso_qom": 1, "sin_match": 0}

def _qom_es_por_fila(row):
    """Devuelve (qom_text, es_text) de la fila según el modo/dirección (sin normalizar)."""
    if MODO_TEXTO == "rol":
        s, r = row.text_source, row.text_reference
        return (s, r) if row.direction == "qom2es" else (r, s)   # qom2es: src=qom
    if MODO_TEXTO == "idioma":
        return (getattr(row, "qom", None), getattr(row, "es", None))
    return (None, None)

def recuperar_segment_ids(j):
    idx = _indice_qomL_hf()
    by_pair, by_qom, by_es, reg = {}, {}, {}, {}
    for r in idx.itertuples(index=False):
        by_pair.setdefault((r.qom_norm, r.es_norm), r.segment_id)
        by_qom.setdefault(r.qom_norm, set()).add(r.segment_id)
        by_es.setdefault(r.es_norm, set()).add(r.segment_id)
        reg[r.segment_id] = (r.qom, r.es, r.config, r.split)
    universo_qom = list(by_qom)

    def _match(qn, en):
        if qn and en and (qn, en) in by_pair: return by_pair[(qn, en)], "par_exacto", 1.0
        if qn and len(by_qom.get(qn, ())) == 1: return next(iter(by_qom[qn])), "solo_qom", 1.0
        if en and len(by_es.get(en, ())) == 1: return next(iter(by_es[en])), "solo_es", 1.0
        if qn:
            mejor, ms = None, 0.0
            for cand in universo_qom:
                v = SequenceMatcher(None, qn, cand).ratio()
                if v > ms: mejor, ms = cand, v
            if mejor is not None and ms >= UMBRAL_DIFUSO and len(by_qom[mejor]) == 1:
                return next(iter(by_qom[mejor])), "difuso_qom", round(ms, 3)
        return None, "sin_match", float("nan")

    sids, met, sc, gq_l, ge_l = [], [], [], [], []
    for row in j.itertuples(index=False):
        qt, et = _qom_es_por_fila(row)
        qn = normalizar_texto(qt) if pd.notna(qt) else None
        en = normalizar_texto(et) if pd.notna(et) else None
        rA = _match(qn, en)
        rB = _match(en, qn) if MODO_TEXTO == "idioma" else (None, "sin_match", float("nan"))
        best = rB if _PRIO[rB[1]] > _PRIO[rA[1]] else rA
        sid = best[0]
        sids.append(sid); met.append(best[1]); sc.append(best[2])
        g = reg.get(sid, (None, None, None, None))
        gq_l.append(g[0]); ge_l.append(g[1])
    out = j.copy()
    out["segment_id"] = sids; out["match_metodo"] = met; out["match_score"] = sc
    # Texto gold de qomL-hf (qom/es reales, sin swap): lo usa el modo "idioma".
    out["_gold_qom"] = pd.array(gq_l, dtype="string")
    out["_gold_es"]  = pd.array(ge_l, dtype="string")
    return out

if "segment_id" in j.columns and j["segment_id"].notna().all():
    print("El CSV ya trae segment_id: se usa tal cual.")
    j["match_metodo"] = "provisto"; j["match_score"] = 1.0
    j["_gold_qom"] = pd.NA; j["_gold_es"] = pd.NA
elif RECUPERAR_ID and MODO_TEXTO != "ninguno":
    j = recuperar_segment_ids(j)
    print("Recuperación de segment_id (trazabilidad):")
    print(j["match_metodo"].value_counts().to_string())
    n_sin = int(j["segment_id"].isna().sum())
    if n_sin:
        print(f"[nota] {n_sin} fila(s) sin segment_id en qomL-hf. NO se descartan: "
              "el chrF++ no depende del id. (Es esperable si no salieron de un split.)")
else:
    print("Sin recuperación de segment_id (RECUPERAR_ID=False o sin texto).")
    j["segment_id"] = j.get("segment_id", pd.Series([pd.NA] * len(j)))
    j["match_metodo"] = "no_recuperado"; j["match_score"] = float("nan")
    j["_gold_qom"] = pd.NA; j["_gold_es"] = pd.NA

JUICIOS_CON_ID = RESULTS_DIR / "juicios_con_id.csv"
j.to_csv(JUICIOS_CON_ID, index=False)
print(f"Juicios (con segment_id si hubo) -> {JUICIOS_CON_ID}")

In [ ]:
from sacrebleu.metrics import CHRF
chrf_pp = CHRF(word_order=2)   # chrF++ (char_order=6, beta=2)

# ── Texto FUENTE y REFERENCIA por fila ────────────────────────────────────────
if MODO_TEXTO == "rol":
    # Directo del CSV: es lo que usó la lingüista. No depende de qomL-hf.
    j["source"]    = j["text_source"]
    j["reference"] = j["text_reference"]
elif MODO_TEXTO == "idioma":
    # qom/es (posible swap): usamos el GOLD de qomL-hf (qom/es reales) para armar
    # src/ref por dirección. Un match DIFUSO no da referencia fiable -> se descarta
    # (evita comparar contra la oración equivocada). Sin match -> también fuera.
    q2e = j["direction"].eq("qom2es")
    confiable = j["match_metodo"].isin(["par_exacto", "solo_qom", "solo_es", "provisto"])
    j["source"]    = j["_gold_qom"].where(q2e, j["_gold_es"]).where(confiable)
    j["reference"] = j["_gold_es"].where(q2e, j["_gold_qom"]).where(confiable)
    n_sin_ref = int(j["reference"].isna().sum())
    if n_sin_ref:
        print(f"[aviso] {n_sin_ref} fila(s) en modo qom/es sin referencia fiable "
              "(sin match exacto en qomL-hf): quedan fuera. "
              "Usá text_source/text_reference para evitar esta limitación.")
else:
    j["source"] = pd.NA; j["reference"] = pd.NA

# ── Hipótesis del sistema evaluado ────────────────────────────────────────────
if "hyp_inline" in j.columns and j["hyp_inline"].notna().any():
    j["hypothesis"] = j["hyp_inline"].astype("string")
    print(f"Hipótesis desde columna inline: {j['hypothesis'].notna().sum()}/{len(j)} filas.")
else:
    j["hypothesis"] = pd.Series([pd.NA] * len(j), dtype="string")
    print("No hay columna inline de predicción.")

# Atajo opcional: preds de la NB1 (sólo cubren el test común).
if j["hypothesis"].isna().any() and Path(TIDY_CSV).exists() and j["segment_id"].notna().any():
    tidy = pd.read_csv(TIDY_CSV)
    tidy["direction"] = tidy["direction"].map(norm_direction)
    t = (tidy[tidy["system"] == SISTEMA_EVALUADO][["segment_id", "direction", "hypothesis"]]
         .rename(columns={"hypothesis": "_hyp_tidy"}))
    j = j.merge(t, on=["segment_id", "direction"], how="left")
    falta = j["hypothesis"].isna()
    j.loc[falta, "hypothesis"] = j.loc[falta, "_hyp_tidy"].astype("string")
    j = j.drop(columns=["_hyp_tidy"])

print(f"Hipótesis faltantes (requieren correr el modelo): {int(j['hypothesis'].isna().sum())}")

In [ ]:
# ── Correr el modelo v2-nostrat para las hipótesis faltantes ──────────────────
# Genera SOLO las que faltan, sobre el texto FUENTE de cada fila, con las mismas
# etiquetas de idioma y GEN_PARAMS que la Notebook 1. Requiere `transformers` +
# `torch` y el checkpoint en CHECKPOINT_EVALUADO. Idealmente, generación
# determinística (beam search) para reproducir lo que vio la lingüista.
falta = j["hypothesis"].isna()
if not falta.any():
    print("No falta generar hipótesis: se saltea el modelo.")
elif CHECKPOINT_EVALUADO is None:
    print(f"[aviso] Faltan {int(falta.sum())} hipótesis y CHECKPOINT_EVALUADO es None.\n"
          "  → Completá el checkpoint de v2-nostrat en la config, o agregá al CSV de\n"
          "    juicios la columna con la predicción del sistema (COL_PREDICCION_INLINE).")
else:
    import torch
    from transformers import AutoModelForSeq2SeqLM, NllbTokenizer
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Generando {int(falta.sum())} hipótesis en {DEVICE}. GEN_PARAMS={GEN_PARAMS}")

    def _ckpt(direccion):
        return CHECKPOINT_EVALUADO[direccion] if isinstance(CHECKPOINT_EVALUADO, dict) \
               else CHECKPOINT_EVALUADO

    def _load(name):
        tok = NllbTokenizer.from_pretrained(name)
        mdl = AutoModelForSeq2SeqLM.from_pretrained(name).to(DEVICE).eval()
        return mdl, tok

    def _translate(mdl, tok, textos, src_lang, tgt_lang, batch=8):
        tok.src_lang = src_lang
        forced = tok.convert_tokens_to_ids(tgt_lang)
        res = []
        for i in range(0, len(textos), batch):
            b = [str(x) for x in textos[i:i + batch]]
            enc = tok(b, return_tensors="pt", padding=True, truncation=True,
                      max_length=GEN_PARAMS["max_new_tokens"]).to(DEVICE)
            with torch.no_grad():
                gen = mdl.generate(**enc, forced_bos_token_id=forced, **GEN_PARAMS)
            res += tok.batch_decode(gen, skip_special_tokens=True)
        return res

    cache = {}
    for direccion in ["qom2es", "es2qom"]:
        m = falta & j["direction"].eq(direccion)
        if not m.any():
            continue
        src_l, tgt_l = DIRECTION_LANGS[direccion]
        ck = _ckpt(direccion)
        if ck not in cache:
            cache[ck] = _load(ck)
        mdl, tok = cache[ck]
        hyps = _translate(mdl, tok, j.loc[m, "source"].tolist(), LANG[src_l], LANG[tgt_l])
        j.loc[m, "hypothesis"] = pd.array(hyps, dtype="string")
    for mdl, tok in cache.values():
        del mdl, tok
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    print(f"Generación lista. Hipótesis faltantes ahora: {int(j['hypothesis'].isna().sum())}")

In [ ]:
# ── chrF++ por segmento (hyp vs ref) y armado de `datos` ──────────────────────
listo = j.dropna(subset=["hypothesis", "reference"]).copy()
listo = listo[listo["hypothesis"].astype(str).str.len() > 0]
listo["chrf_segment"] = [
    round(chrf_pp.sentence_score(str(h), [str(r)]).score, 4)
    for h, r in zip(listo["hypothesis"], listo["reference"])
]

datos = (listo[["segment_id", "direction", "juicio_humano", "chrf_segment",
                "source", "reference", "hypothesis"]]
         .reset_index(drop=True))
sistema = SISTEMA_EVALUADO   # lo usan las celdas siguientes

n_fuera = len(j) - len(datos)
if n_fuera:
    print(f"[aviso] {n_fuera} fila(s) sin hipótesis quedaron fuera del cruce "
          "(faltó generarlas o venían vacías).")
print(f"Segmentos con chrF++ para el dot plot: {len(datos)} (sistema: {sistema}).")
if len(datos):
    print(datos.groupby(["direction", "juicio_humano"]).size().to_string())
datos.to_csv(RESULTS_DIR / "juicio_vs_chrf.csv", index=False)

## 2.2 — Figura principal (dot plot)

Un panel por dirección:

- eje **Y**: chrF++ del segmento;
- eje **X**: categoría de juicio humano, con **jitter** horizontal;
- **color y forma** de marcador según categoría (paleta apta para daltonismo);
- **mediana** de cada grupo con línea horizontal;
- **puntos individuales visibles**: no boxplot ni violín (con este n no corresponde).

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 300,
    "font.size": 18, "axes.titlesize": 22, "axes.labelsize": 20,
    "xtick.labelsize": 16, "ytick.labelsize": 16, "legend.fontsize": 16,
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.constrained_layout.use": True,
})

# Okabe-Ito: color + forma redundantes por categoría (accesibilidad).
ESTILO_CAT = {
    "correcto":   {"color": "#009E73", "marker": "o"},   # verde, círculo
    "dudoso":     {"color": "#E69F00", "marker": "^"},   # naranja, triángulo
    "incorrecto": {"color": "#D55E00", "marker": "s"},   # bermellón, cuadrado
}
ORDEN_CAT = ["incorrecto", "dudoso", "correcto"]

def guardar(fig, nombre):
    fig.savefig(FIG_DIR / f"{nombre}.pdf", bbox_inches="tight")
    fig.savefig(FIG_DIR / f"{nombre}.png", bbox_inches="tight", dpi=300)
    print(f"  guardada: {nombre}.pdf y .png (300 dpi)")

rng = np.random.default_rng(RANDOM_STATE)
direcciones = [d for d in ["qom2es", "es2qom"] if d in set(datos["direction"])]

fig, axes = plt.subplots(1, len(direcciones), figsize=(7 * len(direcciones), 7),
                         sharey=True, squeeze=False)
axes = axes[0]
for ax, direccion in zip(axes, direcciones):
    d = datos[datos["direction"] == direccion]
    cats = [c for c in ORDEN_CAT if c in set(d["juicio_humano"])]
    cats += [c for c in sorted(set(d["juicio_humano"])) if c not in cats]
    for x, cat in enumerate(cats):
        g = d[d["juicio_humano"] == cat]
        est = ESTILO_CAT.get(cat, {"color": "#0072B2", "marker": "D"})
        jitter = rng.uniform(-0.18, 0.18, size=len(g))
        ax.scatter(np.full(len(g), x) + jitter, g["chrf_segment"],
                   color=est["color"], marker=est["marker"], s=130,
                   edgecolor="white", linewidth=0.7, alpha=0.9, zorder=3)
        med = g["chrf_segment"].median()
        ax.plot([x - 0.28, x + 0.28], [med, med], color="#222222", lw=3, zorder=4)
        ax.text(x, -6, f"n={len(g)}", ha="center", va="top", fontsize=13, color="#555")
    ax.set_xticks(range(len(cats))); ax.set_xticklabels(cats)
    ax.set_title(direccion); ax.set_xlabel("juicio humano")
    ax.set_ylim(-8, 105)
axes[0].set_ylabel("chrF++ (segmento)")
fig.suptitle(f"chrF++ por segmento vs juicio humano — {sistema}")
guardar(fig, "figura_dotplot_juicio_chrf")
plt.show()

## 2.3 — Estadística descriptiva, con cautela

- **n por categoría y dirección.** Si alguna categoría tiene **menos de 10 ítems**, no se
  calculan coeficientes de correlación ni tests de hipótesis: se imprime una advertencia
  explícita.
- Sí se reportan **mediana, rango y rango intercuartílico** de chrF++ por categoría.
- Toda medida de asociación que se calcule va con su **IC bootstrap**.

In [ ]:
# ── Descriptivos por dirección x categoría ────────────────────────────────────
filas = []
for direccion in direcciones:
    d = datos[datos["direction"] == direccion]
    for cat, g in d.groupby("juicio_humano"):
        x = g["chrf_segment"]
        filas.append({"direction": direccion, "categoria": cat, "n": len(g),
                      "mediana": round(x.median(), 2), "min": round(x.min(), 2),
                      "max": round(x.max(), 2), "q1": round(x.quantile(0.25), 2),
                      "q3": round(x.quantile(0.75), 2),
                      "iqr": round(x.quantile(0.75) - x.quantile(0.25), 2)})
descriptivos = pd.DataFrame(filas).sort_values(["direction", "categoria"])
print(descriptivos.to_string(index=False))
descriptivos.to_csv(RESULTS_DIR / "descriptivos_por_categoria.csv", index=False)

In [ ]:
# ── Asociación juicio-chrF++, sólo si el n lo permite ─────────────────────────
# Codificamos el juicio ordinalmente SÓLO para medir asociación monotónica (Spearman);
# esto no implica inventar un escalar de calidad, es un rango de las 3 categorías.
# Spearman = Pearson sobre rangos; lo implementamos con numpy para no sumar scipy.
ORDEN_ORDINAL = {"incorrecto": 0, "dudoso": 1, "correcto": 2}

def _rankdata(a):
    a = np.asarray(a, dtype=float)
    orden = a.argsort(kind="mergesort")
    r = np.empty(len(a), dtype=float)
    sa = a[orden]
    i = 0
    while i < len(a):
        j = i
        while j + 1 < len(a) and sa[j + 1] == sa[i]:
            j += 1
        r[orden[i:j + 1]] = (i + j) / 2.0 + 1.0
        i = j + 1
    return r

def spearman(x, y):
    rx, ry = _rankdata(x), _rankdata(y)
    if rx.std() == 0 or ry.std() == 0:
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])

def spearman_ic(x, y, n_boot, rng):
    rho = spearman(x, y)
    idx = np.arange(len(x))
    reps = []
    for _ in range(n_boot):
        s = rng.choice(idx, size=len(idx), replace=True)
        r = spearman(x[s], y[s])
        if not np.isnan(r):
            reps.append(r)
    lo, hi = np.percentile(reps, [2.5, 97.5]) if reps else (np.nan, np.nan)
    return rho, lo, hi

rng_b = np.random.default_rng(RANDOM_STATE + 7)
filas_assoc = []
for direccion in direcciones:
    d = datos[datos["direction"] == direccion]
    conteos = d["juicio_humano"].value_counts()
    categorias_ok = all(conteos.get(c, 0) >= N_MINIMO_TESTS for c in ORDEN_ORDINAL)
    usables = d[d["juicio_humano"].isin(ORDEN_ORDINAL)]
    if not categorias_ok:
        chicas = {c: int(conteos.get(c, 0)) for c in ORDEN_ORDINAL}
        print(f"[{direccion}] n por categoría {chicas}: alguna < {N_MINIMO_TESTS}. "
              "NO se calculan correlaciones ni tests de hipótesis "
              "— el tamaño de muestra no lo permite.")
        continue
    x = usables["juicio_humano"].map(ORDEN_ORDINAL).to_numpy()
    y = usables["chrf_segment"].to_numpy()
    rho, lo, hi = spearman_ic(x, y, N_BOOTSTRAP, rng_b)
    filas_assoc.append({"direction": direccion, "spearman_rho": round(rho, 3),
                        "ic_low": round(lo, 3), "ic_high": round(hi, 3),
                        "n": len(usables)})

if filas_assoc:
    assoc = pd.DataFrame(filas_assoc)
    print(assoc.to_string(index=False))
    assoc.to_csv(RESULTS_DIR / "asociacion_spearman.csv", index=False)
else:
    print("No se calcularon medidas de asociación (ver avisos arriba).")

## 2.4 — Casos para inspección cualitativa

Segmentos **juzgados correctos con chrF++ más bajo** y **juzgados incorrectos con chrF++
más alto** (5 de cada uno por dirección), con fuente, referencia e hipótesis. Sirven como
ejemplos para el póster (dónde la métrica y el juicio humano se contradicen).

In [ ]:
# ── Casos donde métrica y juicio se contradicen (desde `datos`) ───────────────
casos_out = []
for direccion in direcciones:
    d = datos[datos["direction"] == direccion]
    correctos   = d[d["juicio_humano"] == "correcto"].nsmallest(5, "chrf_segment")
    incorrectos = d[d["juicio_humano"] == "incorrecto"].nlargest(5, "chrf_segment")
    for etiqueta, sub in [("correcto_chrf_bajo", correctos),
                          ("incorrecto_chrf_alto", incorrectos)]:
        s = sub.copy(); s.insert(0, "caso", etiqueta); casos_out.append(s)

if casos_out:
    casos = pd.concat(casos_out, ignore_index=True)
    cols = [c for c in ["caso", "direction", "segment_id", "juicio_humano", "chrf_segment",
                        "source", "reference", "hypothesis"] if c in casos.columns]
    casos = casos[cols]
    print(casos.to_string(index=False))
    casos.to_csv(RESULTS_DIR / "casos_cualitativos.csv", index=False)
    print(f"\nGuardado -> {RESULTS_DIR / 'casos_cualitativos.csv'}")
else:
    print("Sin casos para inspección (datos vacío).")

### Cierre

- La figura contesta de un vistazo si **incorrectos** y **correctos** se separan o se
  mezclan en el eje chrF++.
- Con estos tamaños de muestra, la estadística es **descriptiva**: correlaciones y tests
  sólo si cada categoría llega a `N_MINIMO_TESTS`, siempre con IC bootstrap.
- No se derivó ningún escalar de calidad a partir de los juicios categóricos.